01 - Data Preprocessing

This notebook prepares the COVID-19 Romania dataset for analysis.

The main steps are:
1. Load the raw data
2. Clean the county dataset
3. Clean the country dataset
4. Create date features
5. Create a risk level for machine learning
6. Save the processed datasets

In [1]:
import os
import pandas as pd

raw_folder = "../data/raw"
processed_folder = "../data/processed"

os.makedirs(processed_folder, exist_ok=True)

print("Setup completed.")
print("Files found in data/raw:")

for file in os.listdir(raw_folder):
    print(file)

Setup completed.
Files found in data/raw:
ro_covid_19_country_data_time_series.csv
ro_covid_19_time_series.csv


STEP 1 - Load the Raw Data

The dataset contains two CSV files:
1. Country-level COVID-19 time series
2. County-level COVID-19 time series

In [2]:
country_file = os.path.join(raw_folder, "ro_covid_19_country_data_time_series.csv")
county_file = os.path.join(raw_folder, "ro_covid_19_time_series.csv")

country_df = pd.read_csv(country_file)
county_df = pd.read_csv(county_file)

print("Country dataset shape:", country_df.shape)
print("County dataset shape:", county_df.shape)

county_df.head()

Country dataset shape: (113, 8)
County dataset shape: (4624, 4)


,No,County,Confirmed,Date
0,1.,Alba,9,2020-04-02
1,2.,Arad,110,2020-04-02
2,3.,Argeș,10,2020-04-02
3,4.,Bacău,19,2020-04-02
4,5.,Bihor,39,2020-04-02


STEP 2 - Check Columns

This step shows the columns available in both datasets.

In [3]:
print("Country columns:")
print(country_df.columns.tolist())

print("\nCounty columns:")
print(county_df.columns.tolist())

Country columns:
['date', 'ati', 'quarantine', 'isolation', 'tests', 'confirmed', 'recovered', 'deaths']

County columns:
['No', 'County', 'Confirmed', 'Date']


STEP 3 - Clean County Data

The county dataset is cleaned by:
1. Keeping useful columns
2. Converting the date column
3. Converting confirmed cases to numeric format
4. Removing invalid rows

In [4]:
county_clean = county_df[["County", "Confirmed", "Date"]].copy()

county_clean["Date"] = pd.to_datetime(county_clean["Date"], errors="coerce")
county_clean["Confirmed"] = pd.to_numeric(county_clean["Confirmed"], errors="coerce")

county_clean = county_clean.dropna(subset=["County", "Date", "Confirmed"])

county_clean["County"] = county_clean["County"].astype(str).str.strip()
county_clean["Confirmed"] = county_clean["Confirmed"].astype(int)

county_clean = county_clean.sort_values(["County", "Date"])

print("County data cleaned.")
print("Shape:", county_clean.shape)

county_clean.head()

County data cleaned.
Shape: (4624, 3)


,County,Confirmed,Date
0,Alba,9,2020-04-02
42,Alba,13,2020-04-03
84,Alba,15,2020-04-04
127,Alba,33,2020-04-05
169,Alba,34,2020-04-06


STEP 4 - Create Time Features

New columns are created from the date column.

These columns help with time-based analysis.

In [5]:
county_clean["Year"] = county_clean["Date"].dt.year
county_clean["Month"] = county_clean["Date"].dt.month
county_clean["Month_Name"] = county_clean["Date"].dt.month_name()
county_clean["Day"] = county_clean["Date"].dt.day

print("Time features created.")

county_clean.head()

Time features created.


,County,Confirmed,Date,Year,Month,Month_Name,Day
0,Alba,9,2020-04-02,2020,4,April,2
42,Alba,13,2020-04-03,2020,4,April,3
84,Alba,15,2020-04-04,2020,4,April,4
127,Alba,33,2020-04-05,2020,4,April,5
169,Alba,34,2020-04-06,2020,4,April,6


STEP 5 - Create Risk Level for Machine Learning

A risk level is created based on confirmed cases.

The categories are:
1. Low risk, fewer than 500 confirmed cases
2. Medium risk, between 500 and 1999 confirmed cases
3. High risk, 2000 or more confirmed cases

In [6]:
def create_risk_level(value):
    if value < 500:
        return "Low"
    elif value < 2000:
        return "Medium"
    else:
        return "High"

county_clean["Risk_Level"] = county_clean["Confirmed"].apply(create_risk_level)

print("Risk level created.")
print(county_clean["Risk_Level"].value_counts())

county_clean.head()

Risk level created.
Risk_Level
Low       3246
Medium    1227
High       151
Name: count, dtype: int64


,County,Confirmed,Date,Year,Month,Month_Name,Day,Risk_Level
0,Alba,9,2020-04-02,2020,4,April,2,Low
42,Alba,13,2020-04-03,2020,4,April,3,Low
84,Alba,15,2020-04-04,2020,4,April,4,Low
127,Alba,33,2020-04-05,2020,4,April,5,Low
169,Alba,34,2020-04-06,2020,4,April,6,Low


STEP 6 - Clean Country Data

The country-level dataset is cleaned for national trend analysis.

In [7]:
country_clean = country_df.copy()

country_clean["date"] = pd.to_datetime(country_clean["date"], errors="coerce")

numeric_columns = [
    "ati",
    "quarantine",
    "isolation",
    "tests",
    "confirmed",
    "recovered",
    "deaths"
]

for col in numeric_columns:
    country_clean[col] = pd.to_numeric(country_clean[col], errors="coerce")
    country_clean[col] = country_clean[col].fillna(0)

country_clean = country_clean.dropna(subset=["date"])

country_clean["Year"] = country_clean["date"].dt.year
country_clean["Month"] = country_clean["date"].dt.month
country_clean["Month_Name"] = country_clean["date"].dt.month_name()

print("Country data cleaned.")
print("Shape:", country_clean.shape)

country_clean.head()

Country data cleaned.
Shape: (113, 11)


,date,ati,quarantine,isolation,tests,confirmed,recovered,deaths,Year,Month,Month_Name
0,2020-04-02,78,0,114699,28483,2738,267,94,2020,4,April
1,2020-04-03,83,0,114646,31657,3183,283,116,2020,4,April
2,2020-04-04,119,0,113449,36092,3613,329,141,2020,4,April
3,2020-04-05,141,0,108810,38623,3864,374,148,2020,4,April
4,2020-04-06,179,23849,106463,40987,4057,406,157,2020,4,April


STEP 7 - Save Processed Data

The cleaned datasets are saved in the processed folder.

In [8]:
county_clean.to_csv("../data/processed/covid_county_clean.csv", index=False)
country_clean.to_csv("../data/processed/covid_country_clean.csv", index=False)

print("Processed files saved:")
print("../data/processed/covid_county_clean.csv")
print("../data/processed/covid_country_clean.csv")

Processed files saved:
../data/processed/covid_county_clean.csv
../data/processed/covid_country_clean.csv


STEP 8 - Final Check

This step checks that the processed files were created correctly.

In [9]:
print("Files in data/processed:")

for file in os.listdir(processed_folder):
    print(file)

print("\nCounty processed columns:")
print(county_clean.columns.tolist())

Files in data/processed:
covid_country_clean.csv
covid_county_clean.csv

County processed columns:
['County', 'Confirmed', 'Date', 'Year', 'Month', 'Month_Name', 'Day', 'Risk_Level']
